# Domain 2 · Tool Design & MCP Integration (18%)

**How to use this notebook.** One section per task statement (2.1 → 2.5), in guide
order. Each section: the **verbatim guide quote** → a **plain-English table** → a
**runnable cell** that makes the concept observable (a printed tool *choice*, a
structured error, a `tool_choice` mode, a real `.mcp.json`, a real Grep/Glob/Edit) →
**anti-patterns** as commented code → a **pointer to your own code** (`file:line`) →
a **self-check**.

**Running example.** A tiny "research/support" tool registry that grows across the
domain: in 2.1 two *overlapping* tools that misroute, in 2.3 the same tools scoped per
role and driven by `tool_choice`. 2.4 is config (`.mcp.json`), 2.5 is the built-in
Claude Code tools — both shown as real files/operations, not abstractions.

> Setup: uses the `anthropic` SDK and your `ANTHROPIC_API_KEY` from the one `.env` in
> `claude-with-anthropic-api/`. Default model is **Haiku** (these teach the *mechanism*,
> not model IQ). Set `CLAUDE_MODEL=claude-sonnet-4-6` in that `.env` to bump every cell.

In [ ]:
from anthropic import Anthropic
from dotenv import load_dotenv, find_dotenv
from pathlib import Path
import os, json

# Load the key from your ONE .env (the VS Code kernel does NOT source your shell).
_candidates = [
    Path.cwd().parents[1] / "claude-with-anthropic-api" / ".env",
]
_found = find_dotenv(filename='.env', usecwd=True)   # nearest .env walking up (portable)
_envfile = Path(_found) if _found else next((p for p in _candidates if p.exists()), None)
if _envfile:
    load_dotenv(_envfile); print(f"Loaded .env from: {_envfile}")
else:
    print("WARNING: no .env found in", [str(p) for p in _candidates])
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set — check .env path above."

client = Anthropic()
MODEL = os.environ.get("CLAUDE_MODEL") or "claude-haiku-4-5"   # cheap default
print("Using model:", MODEL)

# Anchor paths to the ccaf-prep dir robustly (works whether the kernel's cwd is
# notebooks/ or ccaf-prep/), so the 2.4/2.5 cells can open your real files.
REPO = next((p for p in Path.cwd().resolve().parents if (p / "claude-with-anthropic-api").exists()), Path.cwd())
CCAF_DIR = (REPO / "ccaf-prep") if (REPO / "ccaf-prep").exists() else Path.cwd()
print("ccaf-prep dir:", CCAF_DIR)

---
## 2.1 · Design effective tool interfaces with clear descriptions and boundaries

> **Task Statement 2.1: Design effective tool interfaces with clear descriptions and boundaries**
>
> **Knowledge of:**
> - Tool descriptions as the primary mechanism LLMs use for tool selection; minimal descriptions lead to unreliable selection among similar tools
> - The importance of including input formats, example queries, edge cases, and boundary explanations in tool descriptions
> - How ambiguous or overlapping tool descriptions cause misrouting (e.g., `analyze_content` vs `analyze_document` with near-identical descriptions)
> - The impact of system prompt wording on tool selection: keyword-sensitive instructions can create unintended tool associations
>
> **Skills in:**
> - Writing tool descriptions that clearly differentiate each tool's purpose, expected inputs, outputs, and when to use it versus similar alternatives
> - Renaming tools and updating descriptions to eliminate functional overlap (e.g., renaming `analyze_content` to `extract_web_results` with a web-specific description)
> - Splitting generic tools into purpose-specific tools with defined input/output contracts (e.g., splitting a generic `analyze_document` into `extract_data_points`, `summarize_content`, and `verify_claim_against_source`)
> - Reviewing system prompts for keyword-sensitive instructions that might override well-written tool descriptions

**Plain-English unpacking**

| Guide phrase | What it means concretely |
|---|---|
| Descriptions = **primary** selection mechanism | Claude never sees your function body. It picks a tool from `name` + `description` + `input_schema` **only**. The description *is* the routing logic. |
| Minimal descriptions → unreliable selection | `"Analyzes content."` next to `"Analyzes a document."` gives Claude nothing to disambiguate → it guesses → misroutes. |
| Include input formats / examples / edge cases / **boundaries** | A good description says *what input*, *what it returns*, and **"use this NOT that"**. Boundaries (`"web pages, NOT local files"`) are what stop overlap. |
| Overlapping descriptions cause misrouting | The exam's canonical example: `analyze_content` vs `analyze_document`. The fix is **rename + differentiate**, e.g. → `extract_web_results`. |
| Split generic tools | One `analyze_document` doing everything → split into `extract_data_points` / `summarize_content` / `verify_claim_against_source`, each with a tight contract. |
| System-prompt keyword sensitivity | A keyword in the system prompt (`"always document your steps"`) can pull the model toward a tool named `*document*` regardless of fit. Review prompts for accidental associations. |

**Run it — watch the *description* decide the routing.** Two tools, same job-ish.
First with **vague, overlapping** descriptions, then with **differentiated** ones
(renamed + boundaries). We force a tool with `tool_choice={"type":"any"}` so Claude
*must* pick one — and print which name it chose. The only thing that changes between
the two runs is the **description text**.

In [ ]:
# pyright: reportUndefinedVariable=false   (client / MODEL come from the setup cell)

# The query is unambiguously about a WEB SEARCH RESULTS page (not a stored document).
QUERY = ("Here is the raw HTML of a Google search results page. Pull out the ranked "
         "result items (title, url, snippet) so I can use them.")

def make_tools(good: bool):
    """Same two capabilities; only the descriptions/names differ.
    good=False -> near-identical (the analyze_content vs analyze_document trap).
    good=True  -> renamed + boundaries that differentiate them."""
    if not good:
        return [
            {"name": "analyze_content",  "description": "Analyze content and return insights.",
             "input_schema": {"type": "object", "properties": {"text": {"type": "string"}}}},
            {"name": "analyze_document", "description": "Analyze a document and return insights.",
             "input_schema": {"type": "object", "properties": {"text": {"type": "string"}}}},
        ]
    return [
        {"name": "extract_web_results",
         "description": ("Extract the ranked result items (title, url, snippet) from the raw "
                         "text/HTML of a WEB SEARCH RESULTS page. Input: search-page text. "
                         "Use for live web/search pages — NOT for stored local documents."),
         "input_schema": {"type": "object", "properties": {"page_text": {"type": "string"}},
                          "required": ["page_text"]}},
        {"name": "summarize_document",
         "description": ("Summarize the prose of a stored LOCAL document (report, spec, "
                         "contract) into 3 bullets. Input: document text. Use for saved files "
                         "— NOT for web/search pages."),
         "input_schema": {"type": "object", "properties": {"doc_text": {"type": "string"}},
                          "required": ["doc_text"]}},
    ]

def which_tool(user_text, tools):
    resp = client.messages.create(
        model=MODEL, max_tokens=150, tools=tools,
        tool_choice={"type": "any"},                 # force SOME tool so we always see a choice
        messages=[{"role": "user", "content": user_text}])
    picks = [b.name for b in resp.content if b.type == "tool_use"]
    return resp.stop_reason, picks

stop_v, pick_v = which_tool(QUERY, make_tools(good=False))
print(f"VAGUE/overlapping descriptions -> stop_reason={stop_v}  chose={pick_v}")
print("   (with two near-identical descriptions, the choice is a coin-flip — that is sample Q2.)\n")

stop_g, pick_g = which_tool(QUERY, make_tools(good=True))
print(f"DIFFERENTIATED descriptions    -> stop_reason={stop_g}  chose={pick_g}")
print("   -> 'extract_web_results' wins because its description names the input + the boundary.")

**The anti-patterns (exam distractors).** From the mapping: *#1 build a routing
layer / consolidate prematurely.* Read them as code:

In [ ]:
# ANTI-PATTERN 1: bolt an LLM/keyword ROUTER in front to "fix" misrouting.
#   def route(q):
#       if "document" in q: return "analyze_document"   # brittle keyword rules
#       return "analyze_content"                         # you are re-implementing tool selection
#   -> The model ALREADY does selection from descriptions. Fix the descriptions, not a router.

# ANTI-PATTERN 2: CONSOLIDATE the two tools into one mega-tool to dodge the overlap.
#   {"name": "analyze", "description": "Analyze anything."}   # now it can't express intent/boundaries
#   -> Hides the distinction instead of making it; worse contracts, not better.

# ANTI-PATTERN 3: leave names/descriptions vague and "explain it in the SYSTEM PROMPT".
#   system = "When the user mentions a document, use analyze_document."   # keyword-sensitive
#   -> A stray keyword elsewhere now hijacks selection. Descriptions are the durable fix.

print("Correct (sample Q2): EXPAND + DIFFERENTIATE the descriptions first "
      "(rename analyze_content -> extract_web_results, add input + boundary). Cheapest root-cause fix.")

**In your own code — you already wrote this.** Exercise 1:
`ccaf-prep/exercises/01-support-agent/tools.py`

- `tool_schemas()` at **lines 88–122** builds two deliberately-similar lookup tools
  (`get_customer` vs `lookup_order`); **only the descriptions** disambiguate them
  (`"NOT for order details"` / `"Does NOT identify the customer"` — those are the
  boundaries from the table above).
- The `GOOD_DESCRIPTIONS` toggle at **lines 17–18** (used at line 91): flip it to
  `False` and the descriptions collapse to `"Retrieves customer information."` /
  `"Retrieves order details."` — the analyze_content/analyze_document trap, live.

Open it and map each description clause to a row in the unpacking table.

**Self-check** (cover the answers)

1. What three fields are the *only* things Claude sees when choosing a tool?
2. `analyze_content` and `analyze_document` misroute. What's the cheapest fix — a router, a merge, or rewriting descriptions?
3. Name one thing a good description includes besides "what it does".
4. How can the *system prompt* sabotage well-written tool descriptions?

<details><summary>answers</summary>

1. `name`, `description`, `input_schema` — never the function body.
2. Rewrite/differentiate the descriptions (rename + add boundaries) — sample Q2. A router re-implements selection; a merge destroys the distinction.
3. Input format, an example query, edge cases, and a **boundary** ("use this NOT that").
4. A keyword-sensitive instruction ("when they say *document*…") creates an unintended association that overrides the descriptions.
</details>

---
## 2.2 · Implement structured error responses for MCP tools

> **Task Statement 2.2: Implement structured error responses for MCP tools**
>
> **Knowledge of:**
> - The MCP `isError` flag pattern for communicating tool failures back to the agent
> - The distinction between transient errors (timeouts, service unavailability), validation errors (invalid input), business errors (policy violations), and permission errors
> - Why uniform error responses (generic "Operation failed") prevent the agent from making appropriate recovery decisions
> - The difference between retryable and non-retryable errors, and how returning structured metadata prevents wasted retry attempts
>
> **Skills in:**
> - Returning structured error metadata including `errorCategory` (transient/validation/permission), `isRetryable` boolean, and human-readable descriptions
> - Including `retriable: false` flags and customer-friendly explanations for business rule violations so the agent can communicate appropriately
> - Implementing local error recovery within subagents for transient failures, propagating to the coordinator only errors that cannot be resolved locally along with partial results and what was attempted
> - Distinguishing between access failures (needing retry decisions) and valid empty results (representing successful queries with no matches)

**Plain-English unpacking**

| Guide phrase | What it means concretely |
|---|---|
| `isError` flag | The boolean on the `tool_result` block (`is_error` in the SDK). It tells Claude "this failed" so it doesn't treat garbage as data. |
| transient / validation / business / permission | Four *recovery-relevant* buckets. **transient** = timeout → retry. **validation** = bad input → fix/ask. **business** = policy says no → explain, don't retry. **permission** = not allowed → escalate. |
| Generic "Operation failed" blocks recovery | One opaque string forces the same (wrong) reaction to every failure. The agent can't tell "retry" from "give up". |
| retryable vs non-retryable | An explicit `isRetryable` boolean stops the agent burning turns retrying a `validation`/`business` failure that will never change. |
| customer-friendly explanation on business errors | For policy violations, include a human-readable reason the agent can relay ("refunds over $500 need a manager"). |
| Local recovery in subagents; propagate only the unresolvable | A subagent retries its own transient blips; it escalates to the coordinator **only** what it couldn't fix — with partial results + what it attempted (that's D5.3). |
| access failure ≠ empty result | "DB unreachable" (error → maybe retry) is **not** "query ran, 0 rows" (success → a real, final answer). Don't conflate them. |

**Run it — structured errors let _Claude_ choose recovery (REAL API call).** The whole
point of 2.2 is that the *agent* reacts differently to different failures — so the
decision has to be **Claude's, made from a real call**, not a Python `if`. We give Claude
one `process_refund` tool whose **backend is a mocked fixture** (faking the failing
billing API is fine — it isn't what's being taught), hand the structured error back in a
**real `tool_result`** (`is_error` + `errorCategory` + `isRetryable`), and run the real
agent loop. Watch Claude **retry** a `transient` timeout (then succeed) but **refuse to
retry** a non-retryable `business` rule — it explains instead. Same agent, two error
*shapes*, two recoveries — a generic `"Operation failed"` would collapse both into one
wrong reaction. (The other buckets — `validation`, `permission`, and a *valid empty
result* — show up in the anti-patterns and self-check.)

In [ ]:
# pyright: reportUndefinedVariable=false   (client / MODEL come from the setup cell)

def err(category, retryable, description):
    """The structured error shape (mirrors EX1 tools.py _err)."""
    return {"isError": True, "errorCategory": category,
            "isRetryable": retryable, "description": description}

def tool_result_block(tool_use_id, payload):
    """How the result rides back to Claude — note is_error, exactly like your course
    code (cli_project/core/tools.py:_build_tool_result_part)."""
    return {"type": "tool_result", "tool_use_id": tool_use_id,
            "content": json.dumps(payload), "is_error": bool(payload.get("isError"))}

REFUND_TOOL = {
    "name": "process_refund",
    "description": "Issue a refund for an order. Returns a refund_id on success.",
    "input_schema": {"type": "object",
        "properties": {"order_id": {"type": "string"}, "amount": {"type": "number"}},
        "required": ["order_id", "amount"]},
}

def make_backend(mode):
    """MOCKED billing API (fixture — faking the backend is fine; it's not the lesson).
    transient: times out ONCE, then succeeds.  business: hard, non-retryable NO.
    The structured SHAPE it returns is what Claude actually reacts to."""
    state = {"n": 0}
    def backend(args):
        state["n"] += 1
        if mode == "transient" and state["n"] == 1:
            return err("transient", True, "Billing API timed out — safe to retry.")
        if mode == "business":
            return err("business", False,
                       f"Refund ${args.get('amount')} exceeds the $500 auto-approve "
                       f"limit; a manager must approve.")
        return {"isError": False, "refund_id": "rf_1042", "amount": args.get("amount")}
    return backend

def run_until_done(user_text, mode, max_turns=5):
    """A REAL agent loop: Claude calls the tool, we hand back the STRUCTURED result,
    Claude decides what to do next. We only count calls and capture the final text —
    every recovery decision is Claude's, not ours."""
    backend = make_backend(mode)
    messages = [{"role": "user", "content": user_text}]
    calls = 0
    for _ in range(max_turns):
        resp = client.messages.create(model=MODEL, max_tokens=400,
                                      tools=[REFUND_TOOL], messages=messages)
        # The loop is driven by the STRUCTURED stop_reason (D1.1), not by parsing text
        # or a turn-count: keep going ONLY while Claude wants a tool. Any other reason —
        # end_turn (done) and also max_tokens/refusal/etc. — exits cleanly. (Inverse of
        # "loop while stop_reason == 'tool_use'"; safer than gating on == 'end_turn',
        # which would spin forever on a max_tokens/refusal stop.)
        if resp.stop_reason != "tool_use":
            final = " ".join(b.text for b in resp.content if b.type == "text").strip()
            return calls, final
        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for b in resp.content:
            if b.type == "tool_use":
                calls += 1
                results.append(tool_result_block(b.id, backend(b.input)))
        messages.append({"role": "user", "content": results})
    return calls, "(hit max_turns)"

calls_t, final_t = run_until_done("Please refund $120 on order A-1001.", mode="transient")
print(f"TRANSIENT  (isRetryable=True)  -> process_refund called {calls_t}x  "
      f"=> Claude RETRIED, then succeeded.\n   Claude: {final_t[:150]}\n")

calls_b, final_b = run_until_done("Please refund $900 on order A-2002.", mode="business")
print(f"BUSINESS   (isRetryable=False) -> process_refund called {calls_b}x  "
      f"=> Claude did NOT retry, it EXPLAINED.\n   Claude: {final_b[:150]}\n")

print("Same agent, two error SHAPES, two recoveries — DECIDED BY CLAUDE from the structured "
      "metadata, not a Python switch.\nA generic 'Operation failed' would erase the difference.")

**The anti-patterns (exam distractors).** From the mapping: *generic "operation
failed".* Read them as code:

In [ ]:
# ANTI-PATTERN 1: one opaque string for every failure.
#   return {"isError": True, "description": "Operation failed."}
#   -> The agent can't tell transient(retry) from validation(ask) from business(explain).

# ANTI-PATTERN 2: raise/return a bare exception and let it bubble.
#   raise RuntimeError("boom")        # becomes an untyped is_error with no category
#   -> No isRetryable, no human-readable reason -> the agent guesses.

# ANTI-PATTERN 3: treat a VALID EMPTY RESULT as an error.
#   if not orders: return err("validation", True, "no orders")   # WRONG
#   -> "0 matches" is a successful answer; flagging it as error triggers pointless retries.

# ANTI-PATTERN 4: mark a business/validation failure retryable to "try harder".
#   return err("business", True, "over limit")   # isRetryable=True burns turns on a hard NO.

print("Correct: structured metadata -> errorCategory + isRetryable + human-readable description. "
      "Subagent recovers transient locally; propagates only the unresolvable (with partials).")

**In your own code — you already wrote this.** Two anchors:

- **Exercise 1:** `ccaf-prep/exercises/01-support-agent/tools.py` — `_err(category, retryable,
  description)` at **lines 32–35** is the exact shape above; every failure path
  (**lines 38–82**) returns it with the right category (`validation`, `business`), and
  the multi-match case (**lines 49–53**) asks for an identifier instead of guessing.
  `process_refund` (**65–75**) returns a `business` error when the refund exceeds the
  total.
- **Course code:** `claude-with-anthropic-api/cli_project/core/tools.py` —
  `_build_tool_result_part` at **lines 38–50** sets the `is_error` field (line **49**);
  `execute_tool_requests` (**88–104**) propagates `tool_output.isError` from the MCP
  result into that flag. That's the `isError` → `is_error` bridge, for real.

Map the four categories to the `_err(...)` calls in `tools.py`.

**Self-check** (cover the answers)

1. Which `tool_result` field tells Claude the call failed?
2. A tool returns `errorCategory="business", isRetryable=False`. Should the agent retry? What should it do?
3. "DB connection refused" vs "query returned 0 rows" — which is an error, which is a valid result?
4. Where should a *transient* failure ideally be handled in a coordinator–subagent system?

<details><summary>answers</summary>

1. `is_error` (the SDK form of MCP's `isError`).
2. No — `business` is non-retryable; relay the customer-friendly reason and/or escalate.
3. Connection refused = **access failure** (error, maybe retry). 0 rows = **valid empty result** (success, final answer).
4. Locally, inside the subagent (retry there); propagate to the coordinator only what it can't resolve, with partial results + what it attempted (D5.3).
</details>

---
## 2.3 · Distribute tools appropriately across agents and configure tool choice

> **Task Statement 2.3: Distribute tools appropriately across agents and configure tool choice**
>
> **Knowledge of:**
> - The principle that giving an agent access to too many tools (e.g., 18 instead of 4-5) degrades tool selection reliability by increasing decision complexity
> - Why agents with tools outside their specialization tend to misuse them (e.g., a synthesis agent attempting web searches)
> - Scoped tool access: giving agents only the tools needed for their role, with limited cross-role tools for specific high-frequency needs
> - `tool_choice` configuration options: `"auto"`, `"any"`, and forced tool selection (`{"type": "tool", "name": "..."}`)
>
> **Skills in:**
> - Restricting each subagent's tool set to those relevant to its role, preventing cross-specialization misuse
> - Replacing generic tools with constrained alternatives (e.g., replacing `fetch_url` with `load_document` that validates document URLs)
> - Providing scoped cross-role tools for high-frequency needs (e.g., a `verify_fact` tool for the synthesis agent) while routing complex cases through the coordinator
> - Using `tool_choice` forced selection to ensure a specific tool is called first (e.g., forcing `extract_metadata` before enrichment tools), then processing subsequent steps in follow-up turns
> - Setting `tool_choice: "any"` to guarantee the model calls a tool rather than returning conversational text

**Plain-English unpacking**

| Guide phrase | What it means concretely |
|---|---|
| 18 tools vs 4–5 degrades selection | More tools = a harder choice every turn = more misrouting. Fewer, well-scoped tools per agent = reliable selection. |
| Out-of-specialization tools get misused | Give a *synthesis* agent web search and it'll wander off searching instead of synthesizing. Don't hand it tools its role shouldn't touch. |
| Scoped access + limited cross-role tools | Each agent gets only its role's tools, **plus** a narrow cross-role tool for a common need (a `verify_fact` for the synthesizer) — complex cases still route through the coordinator. (sample Q9) |
| Replace generic with constrained | `fetch_url` (anything) → `load_document` (validates it's a doc URL). Narrower contract = less misuse. |
| `tool_choice: "auto"` | Model **may** call a tool or **may** just talk. Default. |
| `tool_choice: "any"` | Model **must** call *some* tool (no plain text). Use when text isn't an acceptable answer. |
| `tool_choice: {"type":"tool","name":...}` | **Force** this specific tool first (e.g. `extract_metadata` before enrichment), then continue in follow-up turns. |

**Run it — the three `tool_choice` modes, live.** Same toolset, three policies.
Watch `stop_reason` and what Claude returns: `auto` on a greeting → plain text
(`end_turn`); `any` on a task → it's *forced* to call something (`tool_use`); forced
→ it calls **exactly** the tool you named. Reuses the differentiated tools from 2.1.

In [ ]:
# pyright: reportUndefinedVariable=false
TOOLS_23 = make_tools(good=True)          # extract_web_results + summarize_document (from 2.1)
FORCED_NAME = TOOLS_23[0]["name"]         # "extract_web_results"

def run_mode(label, user_text, tool_choice):
    resp = client.messages.create(model=MODEL, max_tokens=150, tools=TOOLS_23,
                                  tool_choice=tool_choice,
                                  messages=[{"role": "user", "content": user_text}])
    picks = [b.name for b in resp.content if b.type == "tool_use"]
    texts = [b.text[:40] for b in resp.content if b.type == "text"]
    print(f"{label:36} stop_reason={resp.stop_reason:9} tools={picks or '—'} text={texts or '—'}")

run_mode('auto + greeting ("hi there!")',     "hi there!",
         {"type": "auto"})
run_mode('any  + task (must call a tool)',     "Pull the result items out of this search page HTML.",
         {"type": "any"})
run_mode(f'forced -> {FORCED_NAME}',           "Do whatever you think best with this text.",
         {"type": "tool", "name": FORCED_NAME})
print("\nauto MAY talk; any MUST call SOME tool; forced calls THAT tool (then you continue in follow-up turns).")

**Run it — scoped distribution (least privilege).** The reliability fix isn't a
bigger model; it's **fewer, role-scoped tools per agent**, with one narrow cross-role
tool for the high-frequency need. This is sample Q9: the synthesis agent gets a scoped
`verify_fact`, **not** the full web-search toolbox.

In [ ]:
# A flat "give everyone everything" registry (the 18-tools-degrade-selection trap):
ALL_TOOLS = ["web_search", "fetch_url", "extract_web_results", "load_document",
             "summarize_document", "verify_fact", "extract_metadata", "enrich_record",
             "write_report", "cite_source"]

# Scoped per role: each agent sees ONLY what its role needs (+ ONE cross-role tool).
ROLE_TOOLS = {
    "researcher":  ["web_search", "extract_web_results", "load_document", "extract_metadata"],
    "synthesizer": ["summarize_document", "write_report", "cite_source",
                    "verify_fact"],   # <- the ONE scoped cross-role tool (sample Q9)
}

print(f"Flat registry: {len(ALL_TOOLS)} tools for EVERY agent -> harder choice every turn.\n")
for role, tools in ROLE_TOOLS.items():
    print(f"  {role:11} sees {len(tools)} tools: {tools}")
print("\nsynthesizer can verify_fact (high-frequency need) but has NO web_search/fetch_url")
print("-> it can't wander off searching; complex research routes back through the coordinator.")

**The anti-patterns (exam distractors).** From the mapping: *#1 give synthesis
all the tools.* Read them as code:

In [ ]:
# ANTI-PATTERN 1 (sample Q9): give the synthesis agent the FULL toolbox "just in case".
#   synthesizer_tools = ALL_TOOLS          # 10-18 tools -> it attempts web_search instead of synthesizing
#   -> Out-of-specialization tools get misused. Scope to role + ONE cross-role verify_fact.

# ANTI-PATTERN 2: always leave tool_choice on "auto" and hope.
#   tool_choice = {"type": "auto"}         # when you NEED a structured tool call, it may return prose
#   -> Use "any" (must call something) or forced (call THIS) when text is not an acceptable answer.

# ANTI-PATTERN 3: keep a generic fetch_url that grabs ANY url.
#   {"name": "fetch_url", "description": "Fetch any URL."}    # invites scraping random pages
#   -> Replace with a constrained load_document that validates it's a document URL.

# ANTI-PATTERN 4: "fix" misrouting with a bigger model instead of fewer tools.
print("Correct (Q9): scope tools per role (4-5, not 18); one narrow cross-role tool for the "
      "high-frequency case; use any/forced tool_choice when text won't do.")

**In your own code — now built (real SDK).** Two anchors:

- **Exercise 4:** `ccaf-prep/exercises/04-multi-agent-research/coordinator_sdk.py` — each
  subagent is an `AgentDefinition` scoped to its **own** tool (**lines 51–60**: the
  `weather` subagent has `tools=[W]`, the `landmark` subagent `tools=[L]` — the SDK
  enforces it, so neither can call the other's tool). `allowed_tools` at **line 73** is
  the *global* execution gate. That's role-scoping as a real mechanism (D1.3 / D2.3).
- **Exercise 3:** `ccaf-prep/exercises/03-extraction-pipeline/extract.py` — `force_extract`
  (**lines 166–183**) uses `tool_choice={"type":"tool","name":"record_refund_request"}`
  (**line 181**) to **force** the extraction tool; the docstring (**169–173**)
  enumerates `auto` / `any` / forced. The three modes you just ran, in your own code.

Open both and match each to a row in the unpacking table.

**Self-check** (cover the answers)

1. An agent misroutes among 18 tools. Bigger model, or fewer tools? Why?
2. Which `tool_choice` mode guarantees Claude calls *some* tool instead of replying in prose?
3. The synthesis agent occasionally needs to check a fact. Give it full web search, or what?
4. How do you force `extract_metadata` to run *before* enrichment?

<details><summary>answers</summary>

1. Fewer tools — decision complexity, not model IQ, is degrading selection (sample Q12's cousin; here Q9).
2. `tool_choice: "any"`. (Forced `{"type":"tool","name":...}` picks a *specific* one.)
3. A scoped cross-role `verify_fact` tool — not the full toolbox; complex cases route through the coordinator (sample Q9).
4. `tool_choice={"type":"tool","name":"extract_metadata"}` on the first turn, then continue (enrichment) in follow-up turns.
</details>

---
## 2.4 · Integrate MCP servers into Claude Code and agent workflows

> **Task Statement 2.4: Integrate MCP servers into Claude Code and agent workflows**
>
> **Knowledge of:**
> - MCP server scoping: project-level (`.mcp.json`) for shared team tooling vs user-level (`~/.claude.json`) for personal/experimental servers
> - Environment variable expansion in `.mcp.json` (e.g., `${GITHUB_TOKEN}`) for credential management without committing secrets
> - That tools from all configured MCP servers are discovered at connection time and available simultaneously to the agent
> - MCP resources as a mechanism for exposing content catalogs (e.g., issue summaries, documentation hierarchies, database schemas) to reduce exploratory tool calls
>
> **Skills in:**
> - Configuring shared MCP servers in project-scoped `.mcp.json` with environment variable expansion for authentication tokens
> - Configuring personal/experimental MCP servers in user-scoped `~/.claude.json`
> - Enhancing MCP tool descriptions to explain capabilities and outputs in detail, preventing the agent from preferring built-in tools (like Grep) over more capable MCP tools
> - Choosing existing community MCP servers over custom implementations for standard integrations (e.g., Jira), reserving custom servers for team-specific workflows
> - Exposing content catalogs as MCP resources to give agents visibility into available data without requiring exploratory tool calls

**Plain-English unpacking**

| Guide phrase | What it means concretely |
|---|---|
| `.mcp.json` (project) vs `~/.claude.json` (user) | Project file = **shared, committed** team tooling. User file = **personal/experimental** servers, not shared via VCS. Scope by audience. |
| `${GITHUB_TOKEN}` expansion | Put a placeholder in `.mcp.json`; the value comes from your environment at runtime. The secret is **never committed**. |
| Discovered at connection time, available simultaneously | At startup the agent connects to every configured server and sees **all** their tools at once — no manual per-call wiring. |
| Resources = content catalogs | A `resource` (e.g. `docs://documents`) hands the agent an **index** of what exists, so it doesn't burn tool calls exploring. |
| Enhance descriptions so MCP tools beat built-ins | If your MCP tool's description is thin, the agent falls back to **Grep**. Describe capabilities/outputs so it prefers the richer tool. |
| Community server over custom | For standard integrations (Jira, GitHub) use the existing community server; reserve custom servers for team-specific workflows. |

**Run it — your REAL `.mcp.json` and a REAL MCP resource.** This task is config,
so the observable is the **files you actually created**: parse Exercise 2's
`.mcp.json` (project scope, `${GITHUB_TOKEN}` expansion) and read the `docs://` resource
in your course MCP server. Nothing is mocked — these are the real artifacts.

> **Note — the `github` server here is *illustrative*.** This cell only *reads the text*
> of `.mcp.json` to show the `${GITHUB_TOKEN}` expansion pattern; it never launches the
> github server. `GITHUB_TOKEN` is not set anywhere in this project, so the server would
> not actually authenticate. The `${GITHUB_TOKEN}` placeholder is resolved from the
> **process environment** of whatever launches the server (Claude Code) — *not* from a
> `.env` file directly. A `.env` only contributes if something loads it into the
> environment first (`direnv`, `set -a; source .env`, `python-dotenv`, …). Same
> placeholder, different source per environment: your shell locally, a GitHub Actions
> secret in CI, a secrets manager on a server. The committed file never changes.

In [ ]:
import re

# --- 1) The REAL project-scoped .mcp.json (Exercise 2) ---
mcp_path = CCAF_DIR / "exercises/02-claude-code-team" / ".mcp.json"
cfg = json.loads(mcp_path.read_text())
print(f"PROJECT-scoped (committed, shared): {mcp_path}")
for name, server in cfg["mcpServers"].items():
    cmd = server.get("command")
    env = server.get("env", {})
    print(f"  server '{name}': command={cmd}  env={env or '—'}")

# Find the ${ENV} placeholder -> the secret is NOT in the file.
raw = mcp_path.read_text()
placeholders = re.findall(r"\$\{(\w+)\}", raw)
print(f"\n  env-var expansion placeholders: {placeholders}  "
      f"-> the real token lives in your shell env, never committed.")
print("  user-scoped personal/experimental servers would instead go in ~/.claude.json (NOT shared).")

# --- 2) A REAL MCP resource exposed as a content catalog ---
server_py = (REPO / "claude-with-anthropic-api" / "cli_project" / "mcp_server.py").read_text()
resources = re.findall(r'@mcp\.resource\(\s*"([^"]+)"', server_py)
print(f"\nMCP resources in cli_project/mcp_server.py (catalogs, not tool calls): {resources}")
print('  docs://documents returns the list of doc ids -> the agent sees the catalog WITHOUT exploring.')

**The anti-patterns (exam distractors).** From the mapping: *commit secrets /
custom over community.* As code/config:

In [ ]:
# ANTI-PATTERN 1: hardcode the secret in .mcp.json (and commit it).
#   {"env": {"GITHUB_PERSONAL_ACCESS_TOKEN": "ghp_EXAMPLE_NOT_REAL"}}   # leaked in VCS forever
#   -> Use "${GITHUB_TOKEN}" expansion; keep the value in the environment.

# ANTI-PATTERN 2: build a CUSTOM Jira/GitHub MCP server from scratch.
#   # 300 lines re-implementing the community @modelcontextprotocol/server-github
#   -> Use the community server for standard integrations; custom only for team-specific workflows.

# ANTI-PATTERN 3: put SHARED team tooling in user-scoped ~/.claude.json.
#   # teammates don't get it; "works on my machine"
#   -> Shared tooling belongs in project .mcp.json (committed). ~/.claude.json is personal/experimental.

# ANTI-PATTERN 4: ship a thin MCP tool description, then wonder why the agent uses Grep instead.
print("Correct: project .mcp.json + ${ENV} expansion for shared tooling; ~/.claude.json for personal; "
      "community servers for standard integrations; resources expose catalogs to cut exploratory calls.")

**In your own code — you already built this.** Two anchors:

- **Exercise 2:** `ccaf-prep/exercises/02-claude-code-team/.mcp.json` (**lines 1–21**) — a
  project-scoped config with a `docs` server (local `uv run` command) and a `github`
  server using `${GITHUB_TOKEN}` expansion (**lines 16–18**). This is the *shared,
  committed* scope.
- **Course code:** `claude-with-anthropic-api/cli_project/mcp_server.py` — the
  `docs://documents` and `docs://documents/{doc_id}` **resources** at **lines 44–60**
  (catalog + fetch), alongside the `read_doc_contents` tool (**17–27**). The resource is
  the *catalog* that saves exploratory calls.

Open `.mcp.json` and point to the one line that keeps your token out of VCS.

**Self-check** (cover the answers)

1. Shared team MCP server: `.mcp.json` or `~/.claude.json`? Personal experiment?
2. How do you reference a `GITHUB_TOKEN` in `.mcp.json` without committing the secret?
3. Your MCP tool exists but the agent keeps using Grep instead. What's the fix?
4. What problem do MCP *resources* solve that tools don't?

<details><summary>answers</summary>

1. Shared → project `.mcp.json` (committed). Personal/experimental → user `~/.claude.json` (not shared).
2. `${GITHUB_TOKEN}` placeholder; the value is expanded from the environment at runtime.
3. Enhance the MCP tool's **description** (capabilities + outputs) so the agent prefers it over the built-in.
4. They expose a **catalog/index** of available content, so the agent doesn't burn exploratory tool calls discovering what exists.
</details>

---
## 2.5 · Select and apply built-in tools (Read, Write, Edit, Bash, Grep, Glob) effectively

> **Task Statement 2.5: Select and apply built-in tools (Read, Write, Edit, Bash, Grep, Glob) effectively**
>
> **Knowledge of:**
> - Grep for content search (searching file contents for patterns like function names, error messages, or import statements)
> - Glob for file path pattern matching (finding files by name or extension patterns)
> - Read/Write for full file operations; Edit for targeted modifications using unique text matching
> - When Edit fails due to non-unique text matches, using Read + Write as a fallback for reliable file modifications
>
> **Skills in:**
> - Selecting Grep for searching code content across a codebase (e.g., finding all callers of a function, locating error messages)
> - Selecting Glob for finding files matching naming patterns (e.g., `**/*.test.tsx`)
> - Using Read to load full file contents followed by Write when Edit cannot find unique anchor text
> - Building codebase understanding incrementally: starting with Grep to find entry points, then using Read to follow imports and trace flows, rather than reading all files upfront
> - Tracing function usage across wrapper modules by first identifying all exported names, then searching for each name across the codebase

**Plain-English unpacking**

| Guide phrase | What it means concretely |
|---|---|
| **Grep** = content search | "Where is this string?" Search *inside* files for a pattern (function name, error message, import). |
| **Glob** = path matching | "Which files match this name?" Find files by name/extension (`**/*.test.tsx`) — it doesn't look inside them. |
| **Bash** = run a shell command | The escape hatch: run a real shell command (`wc -l`, `git log`, `ruff`, a test runner) when no file tool fits — it's how Claude builds, tests, and lints. A shell can do *anything*, so it's the built-in you scope tightest under least privilege (`Bash(wc:*)` rather than blanket `Bash`). |
| **Read/Write** = full file; **Edit** = targeted | Read loads a whole file; Write replaces it; Edit swaps a **unique** anchor string in place. |
| Edit needs a **unique** match | Edit fails (or is unsafe) when the anchor text appears more than once — it can't tell which one you mean. |
| Edit fails → **Read + Write** fallback | When no unique anchor exists, Read the whole file, transform it in memory, Write it back. |
| Incremental understanding | **Grep** for an entry point → **Read** to follow the imports/flow → repeat. Don't Read the whole tree upfront. |
| Trace across wrappers | List all exported names first, then **Grep** each name across the codebase to find every caller. |

**What does "Grep for the entry point" mean?** It's the search *strategy* the last two
rows describe — the opposite of reading a whole codebase to understand it. You use **Grep
(content search) to _locate_ the one place a flow starts**, then Read outward from there.

The **entry point** is wherever the thing you're tracing *begins*:
- the literal program start — `def main`, `if __name__ == "__main__"`, a route like `@app.post("/refund")`, a CLI handler;
- a symbol's definition — `def process_refund`, `class RefundService`;
- or a symptom you're chasing — an error string the user reported (`"refund exceeds limit"`), a config key, a feature flag.

Then you let the **code itself tell you where to look next** (the imports it pulls in, the
functions it calls), reading each precise slice on demand:

```text
Grep "process_refund"        # 1. find WHERE it lives (1 definition + N callers)
Read tools.py:65-75          # 2. read JUST that function -> it calls _validate_amount()
Grep "_validate_amount"      # 3. follow that thread to its definition
Read ...                     # 4. read that slice, repeat until the flow is clear
```

So **"Grep an entry point, Read what you follow"** = start narrow, expand along real
dependencies — pulling in only the slices on the path you're tracing. That's the exact
opposite of the anti-pattern below (`for f in all_files: Read(f)`), which dumps the whole
tree into context, 90% of it irrelevant, and blows the context window. You can watch the
same shape in the run cell: Claude `Grep`s for `tool_choice` to find *which* file matters
first, instead of `Read`-ing every `.py` to discover it.

**Run it — the REAL built-ins, driven by the Agent SDK (not a Python simulation).**
These six (Read/Write/Edit/Bash/Grep/Glob) are **Claude Code built-ins** — *not* tools
you declare with the raw Messages API like `process_refund` in 2.1–2.3. The **Agent SDK**
ships them and actually **executes** them, so we drive the real engine (`query()`, the
same path as D1 §1.2 / `coordinator_sdk.py`): Claude runs an **incremental** task over
your real `ccaf-prep` tree and we print **which built-in it _selects_ for each job** —
`Glob` for paths, `Grep` for content, `Bash` for a shell command, `Read` for a precise
range — never reading the whole tree. The run is **read-only** (no `Write`/`Edit` in
`allowed_tools`, `permission_mode="default"`, no bypass), so it can't touch your files.

> **`Edit`'s non-unique → Read+Write contract** is shown *deterministically* in the
> second half of the cell, on a throwaway temp file. Why not via the live agent? Because
> a capable model usually **dodges** the failure — it supplies a multi-line anchor or
> `replace_all` and the Edit just succeeds — so the exact rule the exam tests ("anchor
> not unique ⇒ fall back to Read+Write") is clearest demonstrated directly. That's the
> one piece we mock; everything above it is a real SDK call with real tool execution.

In [ ]:
# 2.5 — REAL Claude Code BUILT-IN tools via the Agent SDK (NOT a Python simulation).
# Read/Write/Edit/Bash/Grep/Glob are NOT tools you declare with the raw Messages API —
# they ship with Claude Code / the Agent SDK, which actually EXECUTES them. So we drive the
# real engine: query() runs Claude over your real ccaf-prep tree and we watch which built-in
# it SELECTS for each job. Same SDK path as D1 §1.2 / coordinator_sdk.py.
from claude_agent_sdk import (
    query, ClaudeAgentOptions, AssistantMessage, ToolUseBlock, ResultMessage,
)

# The job each built-in is the RIGHT pick for — the selection lesson, printed live.
JOB = {"Glob": "find files by PATH/name", "Grep": "search file CONTENT",
       "Bash": "run a shell command", "Read": "load a precise range"}

async def run_builtins(prompt, allowed, cwd):
    """Run one real Claude Code turn-loop and return (fired, result). `fired` is the ordered
    list of (tool, key-input) the model actually CHOSE — that's the observable for 2.5.
    Permissioning is REAL: permission_mode='default' + an explicit allowed_tools (no bypass)."""
    options = ClaudeAgentOptions(
        model="haiku",                     # cheap on purpose; the lesson is selection, not IQ
        allowed_tools=allowed,             # explicit allow-list (least privilege) — no bypass
        permission_mode="default",
        cwd=str(cwd),                      # run over YOUR real tree
        max_turns=8)                       # cost backstop
    fired, result = [], None
    async for msg in query(prompt=prompt, options=options):
        if isinstance(msg, AssistantMessage):
            for b in msg.content:
                if isinstance(b, ToolUseBlock):
                    key = (b.input.get("pattern") or b.input.get("command")
                           or b.input.get("file_path") or "")
                    fired.append((b.name, str(key)))
                    print(f"   [Claude chose {b.name:5}] {JOB.get(b.name, '?'):22} <- {str(key)[:60]}")
        elif isinstance(msg, ResultMessage):
            result = msg
    return fired, result

# An incremental task (NO reading whole files up front): each step naturally maps to one tool,
# but Claude still does the SELECTING — we print what it picked.
TASK = (
    "Work INCREMENTALLY over this repo — do NOT read whole files up front.\n"
    "1. Find the exercise Python files by their PATH pattern 'exercises/0*/**/*.py'.\n"
    "2. Search those files' CONTENTS for which ones use `tool_choice`.\n"
    "3. For ONE matching file, run a shell command to count how many lines it has.\n"
    "4. Read just its first ~15 lines, then say in ONE sentence what that file does."
)

# Top-level await: Jupyter runs this directly (a plain .py would need asyncio.run).
fired, result = await run_builtins(TASK, ["Glob", "Grep", "Bash", "Read"], CCAF_DIR)
chose = [t for t, _ in fired]
print(f"\nBuilt-ins Claude SELECTED, in order: {chose}")
print("-> Glob=PATHS, Grep=CONTENT, Bash=shell, Read=precise range — each job routed to its "
      "right built-in, incrementally, never reading the whole tree.")
if result:
    cost = f"${result.total_cost_usd:.4f}" if result.total_cost_usd else "n/a"
    print(f"(real run on haiku: turns={result.num_turns}, cost={cost})")

# --- Edit's UNIQUE-anchor rule + the Read+Write fallback (shown DETERMINISTICALLY) ---
# Editing real files is destructive, so we use a throwaway temp file. NOTE: a live agent
# usually DODGES the non-unique failure (multi-line anchor or replace_all), so the precise
# contract the exam tests is clearest shown directly — this mirrors Claude Code's Edit refusal.
import tempfile

def edit_requires_unique(path, old, new):
    """Claude Code's Edit: refuses unless `old` matches EXACTLY once."""
    text = Path(path).read_text()
    if text.count(old) != 1:
        raise ValueError(f"Edit needs a UNIQUE anchor; '{old}' appears {text.count(old)}x")
    Path(path).write_text(text.replace(old, new))

tmp = Path(tempfile.mkdtemp()) / "config.py"
tmp.write_text("level = 1\nlevel = 1\nname = 'x'\n")        # 'level = 1' appears TWICE
try:
    edit_requires_unique(tmp, "level = 1", "level = 2")     # non-unique -> Edit refuses
except ValueError as e:
    print(f"\nEDIT refused: {e}")
    body = tmp.read_text()                                  # READ the whole file
    tmp.write_text(body.replace("level = 1", "level = 2"))  # transform in memory + WRITE back
    print("Read+Write fallback applied -> " + tmp.read_text().replace(chr(10), " | "))

**The anti-patterns (exam distractors).** From the mapping: *reading all files
upfront.* As code:

In [ ]:
# ANTI-PATTERN 1: Read the WHOLE tree upfront to "understand the codebase".
#   for f in all_files: Read(f)        # blows the context window; most of it is irrelevant
#   -> Grep for the entry point, Read only what you follow. Incremental > exhaustive.

# ANTI-PATTERN 2: use Read to FIND content (then eyeball it).
#   Read("big.py")  # ...scan for the function by hand
#   -> Grep searches content directly; Glob finds files by name. Use the right tool.

# ANTI-PATTERN 3: force Edit on a non-unique anchor (or with a huge context blob).
#   Edit(file, old="level = 1", new="level = 2")   # ambiguous -> fails or edits the wrong line
#   -> When no unique anchor exists, fall back to Read + Write.

# ANTI-PATTERN 4: Glob to search file CONTENTS / Grep to list files by name (swapped).
print("Correct: Grep=content, Glob=paths, Read/Write=full file, Edit=unique anchor "
      "(else Read+Write). Build understanding incrementally from an entry point.")

**In your own code / environment.** This task statement is about the Claude Code
built-ins, so the anchors are config + this very session:

- **Exercise 2:** `ccaf-prep/exercises/02-claude-code-team/CLAUDE.md` — the standards block tells
  the team to *"Prefer the built-in tools deliberately: **Grep** for content, **Glob**
  for paths, **Read/Write/Edit** for files (D2.5). Don't read whole trees up front."*
  That's this task statement written as a project rule.
- **This Claude Code session:** every time you watched me `Grep` for a marker, `Glob`
  for files, then `Read` a precise line range (and fall back to `Read`+`Write` when
  `Edit` couldn't find a unique anchor) — that *was* 2.5. Re-read the tool calls in this
  transcript with the table above in hand.

**Self-check** (cover the answers)

1. Find every caller of `process_refund` across the repo — Grep or Glob?
2. Find all `**/*.test.tsx` files — Grep or Glob?
3. `Edit` fails because the anchor text appears 3 times. What's the reliable fallback?
4. Best way to understand a new codebase — Read everything, or what?

<details><summary>answers</summary>

1. **Grep** — searching file *contents* for the name.
2. **Glob** — matching file *paths/names*; it doesn't look inside.
3. **Read + Write**: load the full file, transform in memory, write it back.
4. Incrementally — **Grep** an entry point, **Read** to follow imports/flow; never Read the whole tree upfront.
</details>

---
## Coming next / related domains

You've made all of Domain 2 observable: descriptions drive selection (2.1), structured
errors drive recovery (2.2), scope + `tool_choice` drive reliability (2.3), `.mcp.json`
+ resources wire it into Claude Code (2.4), and the built-ins pick themselves once you
know Grep=content / Glob=paths (2.5).

**Threads that continue elsewhere:**
- **D1.2 / D1.3** — per-subagent tool scoping as the *real* SDK mechanism
  (`AgentDefinition.tools`) lives in `notebooks/D1_agentic_loops.ipynb` §1.2–1.3 and
  `exercises/04-multi-agent-research/coordinator_sdk.py`.
- **D4.3 / D4.4** — `tool_choice` forced selection for *structured output* and the
  validation-retry loop (where `is_error` carries the feedback) → Domain 4.
- **D5.3** — propagating the *unresolvable* errors (with partials + what was attempted)
  up to the coordinator → Domain 5.

Next domain to build: **D3 — Claude Code Configuration & Workflows (20%)** (CLAUDE.md
hierarchy, slash commands/skills, path rules, plan mode, iterative refinement, CI/CD).

## Sample exam questions — Domain 2
<!-- ccaf:closing -->

Official sample questions whose tested skill lives in this domain. Cover the answer, say why each of the other three is a distractor, then reveal. (You'll meet them again, grouped by scenario, in `mock_exam_and_review.ipynb`.)

**Question 2.** Production logs show the agent frequently calls `get_customer` when users ask about orders (e.g., "check my order #12345"), instead of calling `lookup_order`. Both tools have minimal descriptions ("Retrieves customer information" / "Retrieves order details") and accept similar identifier formats. What's the most effective first step to improve tool selection reliability?

- **A)** Add few-shot examples to the system prompt demonstrating correct tool selection patterns, with 5-8 examples showing order-related queries routing to `lookup_order`.
- **B)** Expand each tool's description to include input formats it handles, example queries, edge cases, and boundaries explaining when to use it versus similar tools.
- **C)** Implement a routing layer that parses user input before each turn and pre-selects the appropriate tool based on detected keywords and identifier patterns.
- **D)** Consolidate both tools into a single `lookup_entity` tool that accepts any identifier and internally determines which backend to query.

<sub>Maps to: D2 §2.1 (tool descriptions)</sub>

<details><summary>Answer + explanation</summary>

**Correct answer: B**

Tool descriptions are the primary mechanism LLMs use for tool selection. When descriptions are minimal, models lack the context to differentiate between similar tools. Option B directly addresses this root cause with a low-effort, high-leverage fix. Few-shot examples (A) add token overhead without fixing the underlying issue. A routing layer (C) is over-engineered and bypasses the LLM's natural language understanding. Consolidating tools (D) is a valid architectural choice but requires more effort than a "first step" warrants when the immediate problem is inadequate descriptions.

</details>

---

**Question 9.** During testing, you observe that the synthesis agent frequently needs to verify specific claims while combining findings. Currently, when verification is needed, the synthesis agent returns control to the coordinator, which invokes the web search agent, then re-invokes synthesis with results. This adds 2-3 round trips per task and increases latency by 40%. Your evaluation shows that 85% of these verifications are simple fact-checks (dates, names, statistics) while 15% require deeper investigation. What's the most effective approach to reduce overhead while maintaining system reliability?

- **A)** Give the synthesis agent a scoped `verify_fact` tool for simple lookups, while complex verifications continue delegating to the web search agent through the coordinator.
- **B)** Have the synthesis agent accumulate all verification needs and return them as a batch to the coordinator at the end of its pass, which then sends them all to the web search agent at once.
- **C)** Give the synthesis agent access to all web search tools so it can handle any verification need directly without round-trips through the coordinator.
- **D)** Have the web search agent proactively cache extra context around each source during initial research, anticipating what the synthesis agent might need to verify.

<sub>Maps to: D2 §2.3 (tool distribution / scoped tools)</sub>

<details><summary>Answer + explanation</summary>

**Correct answer: A**

Option A applies the principle of least privilege by giving the synthesis agent only what it needs for the 85% common case (simple fact verification) while preserving the existing coordination pattern for complex cases. Option B's batching approach creates blocking dependencies since synthesis steps may depend on earlier verified facts. Option C over-provisions the synthesis agent, violating separation of concerns. Option D relies on speculative caching that cannot reliably predict what the synthesis agent will need to verify.

</details>

---


## Exercises that use this domain

Exercises are **cross-domain** — none is D2-only — so do them once all their domains are studied
(see [`README.md`](./README.md) for the wave order).

| Exercise | Domains it reinforces | Do it after you've studied |
|----------|-----------------------|-----------------------------|
| **Ex1** `../exercises/01-support-agent/`        | D1 + D2 + D5 | D1, D2, **D5** |
| **Ex4** `../exercises/04-multi-agent-research/` | D1 + D2 + D5 | D1, D2, **D5** |
| **Ex2** `../exercises/02-claude-code-team/`     | D3 + D2      | D2, **D3**     |

D2 feeds three exercises but none is unlocked yet after this notebook: Ex1/Ex4 wait on **D5**,
Ex2 waits on **D3**.
